In [2]:
import os
import copy
import numpy as np
import pandas as pd
import traceback
import sys
import cupy as cp
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
## change the directory to the wholistic_registration code dir, so that we can import the modules in utils
os.chdir('/home/cyf/wbi/Virginia/code/wbi_0123/wholistic_registration/src/wholistic_registration')
from utils import IO
from utils import calFlowCrossResolution, calFlow3d_Wei_v1
from utils import registration
from utils import option
from utils import preprocess as prep
from utils import mask
from utils import  visualization
import numpy as np
gut_mov_Path     = "/home/cyf/wbi/Virginia/registrated_data/f260201/gut/raw/260201_test1_0_00002_TZCXY.ome.tif"
ventral_mov_Path = "/home/cyf/wbi/Virginia/registrated_data/f260201/ventral/raw/260201_test1_0_00003_TZCXY.ome.tif"
dorsal_mov_Path  = "/home/cyf/wbi/Virginia/registrated_data/f260201/dorsal/raw/260201_test1_0_00005_TZCXY.ome.tif"
gut_ref_Path     = "/home/cyf/wbi/Virginia/registrated_data/f260201/gut/anat/260201_test1_0_00002_TZCXY.ome.tif"
ventral_ref_Path = "/home/cyf/wbi/Virginia/registrated_data/f260201/ventral/anat/260201_test1_0_00002_TZCXY.ome.tif"
dorsal_ref_Path  = "/home/cyf/wbi/Virginia/registrated_data/f260201/dorsal/anat/260201_test1_0_00004_TZCXY.ome.tif"

CuPy is available with CUDA - using GPU acceleration


gut
---

In [3]:
# ------------------------------------------------------------
# Basic config
# ------------------------------------------------------------
out_root = "./HR_exp/gut_exp_sweep_v2"

frames_to_run = [0, 2]
slices_in_crop = range(25, 41)   # 25..40 inclusive
layers_to_run = [1, 2, 3]

ref_frame = 1
ref_z0 = 40
ref_z1 = 120

roi_x0, roi_x1 = 350, 700
roi_y0, roi_y1 = 60, 410

# If you want less terminal output, set False
verbose_run = False

# ------------------------------------------------------------
# Read data
# ------------------------------------------------------------
gut_ref, gut_ref_desc = IO.readTiff(gut_ref_Path)
gut_mov, gut_mov_desc = IO.readTiff(gut_mov_Path)

gut_ref = gut_ref.transpose(1, 2, 0)
gut_mov = gut_mov.transpose(0, 2, 3, 1)

# Fixed reference
gut_ref_sample = gut_mov[ref_frame, :, :, ref_z0:ref_z1]

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def make_case_option(
    base_option,
    gut_ref_sample,
    gut_single_plane,
    slice_in_crop,
    layer,
    wrong_region_enable,
):
    """
    Build an independent option dict for one case.
    Avoid modifying the global option in-place across runs.
    """
    opt = copy.deepcopy(base_option)

    opt["motion"] = None
    opt["r"] = 5
    opt["layer"] = int(layer)
    opt["iter"] = 10
    opt["movRange"] = 15.0
    opt["tol"] = 1e-6

    opt["zRatio"] = 1
    opt["zRatio_HR"] = 3

    opt["mask_ref"] = np.zeros_like(gut_ref_sample, dtype=bool)
    opt["mask_mov"] = np.zeros_like(gut_single_plane, dtype=bool)

    smoothPenalty_raw = 0.03
    Pnltfactor = prep.getSmPnltNormFctr(gut_ref_sample, opt)
    opt["smoothPenalty"] = Pnltfactor * smoothPenalty_raw

    # phase init: z coordinate is slice index inside the cropped reference volume
    H, W, D = gut_single_plane.shape
    X, Y, Z = np.indices((H, W, D))

    phase_init = np.stack(
        [
            X.astype(np.float32),
            Y.astype(np.float32),
            (slice_in_crop + Z * opt["zRatio"] / opt["zRatio_HR"]).astype(np.float32),
        ],
        axis=-1,
    )

    opt["phase"] = np.asarray(phase_init, dtype=np.float32)
    opt["motion"] = None

    opt["wrong_region_enable"] = bool(wrong_region_enable)

    # error correction params
    if wrong_region_enable:
        opt["wrong_region_metric"] = "mae"
        opt["wrong_region_mad_threshold"] = 3
        opt["wrong_region_min_component_size"] = 2
        opt["wrong_region_exclude_mode"] = "highresidual"

    return opt

import gc

def cleanup_memory():
    gc.collect()

    # 如果用了 PyTorch
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

    # 如果用了 CuPy
    try:
        import cupy as cp
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass
    
def roi_mae(src, pred):
    """
    Compute MAE inside ROI [350:700, 60:410].
    Both src and pred should be shape (H, W, 1) or (H, W).
    """
    src2 = np.asarray(src).squeeze()
    pred2 = np.asarray(pred).squeeze()

    src_roi = src2[roi_x0:roi_x1, roi_y0:roi_y1]
    pred_roi = pred2[roi_x0:roi_x1, roi_y0:roi_y1]

    return float(np.mean(np.abs(src_roi - pred_roi)))


def safe_makedirs(path):
    os.makedirs(path, exist_ok=True)


# ------------------------------------------------------------
# Main sweep
# ------------------------------------------------------------
records = []

total_cases = len(frames_to_run) * len(list(slices_in_crop)) * len(layers_to_run)
case_id = 0

for frame in frames_to_run:
    gut_mov_sample = gut_mov[frame, :, :, ref_z0:ref_z1]

    for slice_in_crop in slices_in_crop:
        raw_slice = slice_in_crop + ref_z0

        gut_single_plane = gut_mov_sample[:, :, slice_in_crop:slice_in_crop + 1]

        for layer in layers_to_run:
            case_id += 1

            case_dir = os.path.join(
                out_root,
                f"frame_{frame}",
                f"slice_{raw_slice}",
                f"layer_{layer}",
            )
            safe_makedirs(case_dir)

            print(
                f"\n[{case_id}/{total_cases}] "
                f"frame={frame}, slice_in_crop={slice_in_crop}, raw_slice={raw_slice}, layer={layer}"
            )

            try:
                # ------------------------------------------------------------
                # 1. No correction
                # ------------------------------------------------------------
                opt_nocorr = make_case_option(
                    base_option=option,
                    gut_ref_sample=gut_ref_sample,
                    gut_single_plane=gut_single_plane,
                    slice_in_crop=slice_in_crop,
                    layer=layer,
                    wrong_region_enable=False,
                )

                phase_nocorr, motion_nocorr, mapped_nocorr = calFlowCrossResolution.getMotion_v2(
                    gut_single_plane,
                    gut_ref_sample,
                    opt_nocorr,
                    verbose=verbose_run,
                )

                loss_nocorr_roi = roi_mae(gut_single_plane, mapped_nocorr)

                # ------------------------------------------------------------
                # 2. With error correction
                # ------------------------------------------------------------
                opt_corr = make_case_option(
                    base_option=option,
                    gut_ref_sample=gut_ref_sample,
                    gut_single_plane=gut_single_plane,
                    slice_in_crop=slice_in_crop,
                    layer=layer,
                    wrong_region_enable=True,
                )

                phase_corr, motion_corr, mapped_corr = calFlowCrossResolution.getMotion_v2(
                    gut_single_plane,
                    gut_ref_sample,
                    opt_corr,
                    verbose=verbose_run,
                )

                loss_corr_roi = roi_mae(gut_single_plane, mapped_corr)

                delta_loss = loss_corr_roi - loss_nocorr_roi

                print(
                    f"  ROI loss: "
                    f"nocorr={loss_nocorr_roi:.6f}, "
                    f"corr={loss_corr_roi:.6f}, "
                    f"delta={delta_loss:.6f}"
                )

                # ------------------------------------------------------------
                # 3. Save images
                # ------------------------------------------------------------
                # channel 0: no correction
                # channel 1: correction
                # channel 2: original moving single plane
                IO.write_multichannel_volume_as_ome_tiff(
                    volume=[
                        np.asarray(mapped_nocorr).squeeze(),
                        np.asarray(mapped_corr).squeeze(),
                        np.asarray(gut_single_plane).squeeze(),
                    ],
                    out_dir=case_dir,
                    frame_idx=0,
                    label=f"frame{frame}_slice{raw_slice}_layer{layer}",
                )

                # ------------------------------------------------------------
                # 4. Save metrics and optional arrays
                # ------------------------------------------------------------
                np.savez_compressed(
                    os.path.join(case_dir, "metrics.npz"),
                    frame=frame,
                    slice_in_crop=slice_in_crop,
                    raw_slice=raw_slice,
                    layer=layer,
                    loss_nocorr_roi=loss_nocorr_roi,
                    loss_corr_roi=loss_corr_roi,
                    delta_loss=delta_loss,
                )

                # Optional: save motion/phase if you want to debug later.
                # These files can be large; enable only when needed.
                # np.savez_compressed(
                #     os.path.join(case_dir, "fields.npz"),
                #     phase_nocorr=phase_nocorr,
                #     motion_nocorr=motion_nocorr,
                #     phase_corr=phase_corr,
                #     motion_corr=motion_corr,
                # )

                records.append(
                    {
                        "frame": frame,
                        "slice_in_crop": slice_in_crop,
                        "raw_slice": raw_slice,
                        "layer": layer,
                        "loss_nocorr_roi": loss_nocorr_roi,
                        "loss_corr_roi": loss_corr_roi,
                        "delta_loss": delta_loss,
                        "case_dir": case_dir,
                        "status": "ok",
                    }
                )

            except Exception as e:
                print(
                    f"  ERROR: frame={frame}, raw_slice={raw_slice}, layer={layer}\n"
                    f"  {repr(e)}"
                )

                with open(os.path.join(case_dir, "error.txt"), "w", encoding="utf-8") as f:
                    f.write(traceback.format_exc())

                records.append(
                    {
                        "frame": frame,
                        "slice_in_crop": slice_in_crop,
                        "raw_slice": raw_slice,
                        "layer": layer,
                        "loss_nocorr_roi": np.nan,
                        "loss_corr_roi": np.nan,
                        "delta_loss": np.nan,
                        "case_dir": case_dir,
                        "status": "error",
                    }
                )


# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------



[1/96] frame=0, slice_in_crop=25, raw_slice=65, layer=1
  ROI loss: nocorr=173.353241, corr=173.353241, delta=0.000000

[2/96] frame=0, slice_in_crop=25, raw_slice=65, layer=2
  ROI loss: nocorr=172.930978, corr=172.930978, delta=0.000000

[3/96] frame=0, slice_in_crop=25, raw_slice=65, layer=3
  ROI loss: nocorr=173.460299, corr=173.460299, delta=0.000000

[4/96] frame=0, slice_in_crop=26, raw_slice=66, layer=1
  ROI loss: nocorr=179.318503, corr=178.455699, delta=-0.862804

[5/96] frame=0, slice_in_crop=26, raw_slice=66, layer=2
  ROI loss: nocorr=178.569867, corr=177.229210, delta=-1.340657

[6/96] frame=0, slice_in_crop=26, raw_slice=66, layer=3
  ROI loss: nocorr=178.863058, corr=178.863058, delta=0.000000

[7/96] frame=0, slice_in_crop=27, raw_slice=67, layer=1
  ROI loss: nocorr=185.435071, corr=185.836512, delta=0.401441

[8/96] frame=0, slice_in_crop=27, raw_slice=67, layer=2
  ROI loss: nocorr=185.118174, corr=184.544690, delta=-0.573484

[9/96] frame=0, slice_in_crop=27, ra

In [5]:
df = pd.DataFrame(records)

summary_csv = os.path.join(out_root, "summary_all_cases.csv")
df.to_csv(summary_csv, index=False)

# Cases where correction is best than no correction
df_ok = df[df["status"] == "ok"].copy()
df_best = df_ok[df_ok["delta_loss"] < 0].copy()
df_top20_best = df_best.sort_values("delta_loss", ascending=True).head(20)

top20_csv = os.path.join(out_root, "top20_correction_best_cases.csv")
df_top20_best.to_csv(top20_csv, index=False)

print("\n============================================================")
print("Sweep finished.")
print(f"All summary saved to: {summary_csv}")
print(f"Top-20 best cases saved to: {top20_csv}")
print("============================================================")

print("\nTop-20 cases where error correction is best:")
print(
    df_top20_best[
        [
            "frame",
            "raw_slice",
            "slice_in_crop",
            "layer",
            "loss_nocorr_roi",
            "loss_corr_roi",
            "delta_loss",
            "case_dir",
        ]
    ].to_string(index=False)
)


Sweep finished.
All summary saved to: ./HR_exp/gut_exp_sweep_v2/summary_all_cases.csv
Top-20 best cases saved to: ./HR_exp/gut_exp_sweep_v2/top20_correction_best_cases.csv

Top-20 cases where error correction is best:
 frame  raw_slice  slice_in_crop  layer  loss_nocorr_roi  loss_corr_roi  delta_loss                                           case_dir
     2         73             33      3       198.686626     194.810894   -3.875732 ./HR_exp/gut_exp_sweep_v2/frame_2/slice_73/layer_3
     0         71             31      1       210.955430     208.090469   -2.864961 ./HR_exp/gut_exp_sweep_v2/frame_0/slice_71/layer_1
     2         68             28      3       178.964112     176.668352   -2.295760 ./HR_exp/gut_exp_sweep_v2/frame_2/slice_68/layer_3
     0         72             32      1       213.030019     210.749468   -2.280550 ./HR_exp/gut_exp_sweep_v2/frame_0/slice_72/layer_1
     0         68             28      2       191.293774     189.089912   -2.203863 ./HR_exp/gut_exp_sweep

crop 1 frame 4   slice 68   layer 2  [420:520,130:230]  v1<br> full image [340:540,110:310]
crop 2 frame 2   slice 73   layer 3  [350:450,200:300]  v2<br> full image [340:540,110:310]


F1062
---

In [8]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
import os
import gc
import csv
import traceback
import numpy as np

# 如果你后面要用 pandas，可以保留；没有 pandas 也不影响主流程
try:
    import pandas as pd
except Exception:
    pd = None


# ------------------------------------------------------------
# Basic config
# ------------------------------------------------------------
out_root = "./HR_exp/F1062_exp_sweep_v3"

F1062_path = "/home/cyf/wbi/Virginia/raw_data/f1062/250221_f1062_ubi_bactin_7dpf002.nd2"

frames_to_run = [1, 2, 3, 4]
slices_in_crop = range(5, 35)   # 5..24 inclusive
layers_to_run = [1,2,3]

ref_frame = 0
ref_z0 = 5
ref_z1 = 45

roi_x0, roi_x1 = 0, 600
roi_y0, roi_y1 = 300, 2100

verbose_run = False

# 遇到 cudaErrorIllegalAddress 时建议直接停止，因为 CUDA context 往往已经坏掉
stop_on_cuda_illegal_address = True

# 是否保存 mapped 图像
save_mapped_tiff = True


# ------------------------------------------------------------
# Memory helpers
# ------------------------------------------------------------
def cleanup_memory():
    """
    Release Python / PyTorch / CuPy cached memory as much as possible.
    """
    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

    try:
        import cupy as cp
        cp.cuda.Stream.null.synchronize()
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass


def to_numpy_cpu(x, dtype=np.float32):
    """
    Convert numpy / cupy / torch tensor to CPU numpy array.
    """
    try:
        import cupy as cp
        if isinstance(x, cp.ndarray):
            x = cp.asnumpy(x)
    except Exception:
        pass

    try:
        import torch
        if torch.is_tensor(x):
            x = x.detach().cpu().numpy()
    except Exception:
        pass

    x = np.asarray(x)

    if dtype is not None:
        x = x.astype(dtype, copy=False)

    return np.ascontiguousarray(x)


def safe_delete(*names):
    """
    This function is only a placeholder for readability.
    Real deletion is done by del variable_name in the caller scope.
    """
    cleanup_memory()


def print_mem(prefix=""):
    """
    Optional GPU memory print. Useful for debugging memory leakage.
    """
    msg = prefix

    try:
        import torch
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            msg += f" | torch allocated={alloc:.3f} GB, reserved={reserved:.3f} GB"
    except Exception:
        pass

    try:
        import cupy as cp
        pool = cp.get_default_memory_pool()
        used = pool.used_bytes() / 1024**3
        total = pool.total_bytes() / 1024**3
        msg += f" | cupy used={used:.3f} GB, pool={total:.3f} GB"
    except Exception:
        pass

    if msg:
        print(msg)


# ------------------------------------------------------------
# IO helpers
# ------------------------------------------------------------
def safe_makedirs(path):
    os.makedirs(path, exist_ok=True)


def save_records(records, out_root):
    safe_makedirs(out_root)

    csv_path = os.path.join(out_root, "summary.csv")

    if len(records) == 0:
        return

    fieldnames = list(records[0].keys())

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)

    if pd is not None:
        df = pd.DataFrame(records)
        df.to_csv(csv_path, index=False)

        ok_df = df[df["status"] == "ok"].copy()
        if len(ok_df) > 0:
            ok_df = ok_df.sort_values("delta_loss", ascending=False)
            top20_path = os.path.join(out_root, "top20_corr_worse_than_nocorr.csv")
            ok_df.head(20).to_csv(top20_path, index=False)

    print(f"\nSaved summary to: {csv_path}")


# ------------------------------------------------------------
# Metric
# ------------------------------------------------------------
def roi_mae(src, pred):
    """
    Compute MAE inside ROI.
    Both src and pred can be shape (H, W, 1) or (H, W).
    """
    src2 = to_numpy_cpu(src, dtype=np.float32).squeeze()
    pred2 = to_numpy_cpu(pred, dtype=np.float32).squeeze()

    src_roi = src2[roi_x0:roi_x1, roi_y0:roi_y1]
    pred_roi = pred2[roi_x0:roi_x1, roi_y0:roi_y1]

    return float(np.mean(np.abs(src_roi - pred_roi)))


# ------------------------------------------------------------
# Option builder
# ------------------------------------------------------------
def make_case_option(
    base_option,
    F1062_ref_sample,
    F1062_single_plane,
    slice_in_crop,
    layer,
    wrong_region_enable,
):
    """
    Build an independent option dict for one case.

    Important changes:
    1. Use shallow copy instead of deepcopy.
    2. Remove stale large fields from base_option.
    3. Force phase / masks to be C-contiguous CPU numpy arrays.
    4. Avoid np.indices to reduce temporary memory.
    """

    # 不建议 deepcopy，因为 base_option 里可能带有旧的 phase/motion/cache/GPU 对象
    opt = dict(base_option)

    # 删除可能残留的大对象，避免跨 case 污染
    for k in [
        "phase",
        "motion",
        "mask_ref",
        "mask_mov",
        "mapped",
        "mapped_ref",
        "mapped_mov",
        "cache",
        "cached_ref",
        "cached_mov",
        "gpu_cache",
    ]:
        opt.pop(k, None)

    opt["motion"] = None
    opt["r"] = 5
    opt["layer"] = int(layer)
    opt["iter"] = 10
    opt["movRange"] = 15.0
    opt["tol"] = 1e-6

    opt["zRatio"] = 1
    opt["zRatio_HR"] = 1

    # mask 强制连续
    opt["mask_ref"] = np.ascontiguousarray(
        np.zeros(F1062_ref_sample.shape, dtype=bool)
    )
    opt["mask_mov"] = np.ascontiguousarray(
        np.zeros(F1062_single_plane.shape, dtype=bool)
    )

    smoothPenalty_raw = 0.001
    Pnltfactor = prep.getSmPnltNormFctr(F1062_ref_sample, opt)
    opt["smoothPenalty"] = Pnltfactor * smoothPenalty_raw

    # phase init
    # 不使用 np.indices，减少 int64 临时数组
    H, W, D = F1062_single_plane.shape

    phase_init = np.empty((H, W, D, 3), dtype=np.float32)

    x_coord = np.arange(H, dtype=np.float32)[:, None, None]
    y_coord = np.arange(W, dtype=np.float32)[None, :, None]
    z_coord = np.arange(D, dtype=np.float32)[None, None, :]

    phase_init[..., 0] = x_coord
    phase_init[..., 1] = y_coord
    phase_init[..., 2] = (
        float(slice_in_crop)
        + z_coord * float(opt["zRatio"]) / float(opt["zRatio_HR"])
    )

    opt["phase"] = np.ascontiguousarray(phase_init, dtype=np.float32)
    opt["motion"] = None

    opt["wrong_region_enable"] = bool(wrong_region_enable)

    if wrong_region_enable:
        opt["wrong_region_metric"] = "mae"
        opt["wrong_region_mad_threshold"] = 3.0
        opt["wrong_region_min_component_size"] = 2
        opt["wrong_region_exclude_mode"] = "highresidual"
    else:
        # 避免 base_option 里原本残留这些开关
        opt["motion_inpaint_enable"] = False

    return opt


# ------------------------------------------------------------
# Run one case
# ------------------------------------------------------------
def run_one_case(
    frame,
    slice_in_crop,
    raw_slice,
    layer,
    case_dir,
    F1062_ref_sample,
    F1062_single_plane,
    option,
):
    """
    Run no-correction and correction sequentially.
    The no-correction large variables are released before correction starts.
    """

    result = {
        "frame": frame,
        "slice_in_crop": slice_in_crop,
        "raw_slice": raw_slice,
        "layer": layer,
        "loss_nocorr_roi": np.nan,
        "loss_corr_roi": np.nan,
        "delta_loss": np.nan,
        "case_dir": case_dir,
        "status": "error",
        "error": "",
    }

    mapped_nocorr_save = None
    mapped_corr_save = None

    # ------------------------------------------------------------
    # 1. No correction
    # ------------------------------------------------------------
    opt_nocorr = make_case_option(
        base_option=option,
        F1062_ref_sample=F1062_ref_sample,
        F1062_single_plane=F1062_single_plane,
        slice_in_crop=slice_in_crop,
        layer=layer,
        wrong_region_enable=False,
    )

    phase_nocorr, motion_nocorr, mapped_nocorr = calFlowCrossResolution.getMotion_v2(
        F1062_single_plane,
        F1062_ref_sample,
        opt_nocorr,
        verbose=verbose_run,
    )

    mapped_nocorr_save = to_numpy_cpu(mapped_nocorr, dtype=np.float32).squeeze()
    loss_nocorr_roi = roi_mae(F1062_single_plane, mapped_nocorr_save)

    # 不保存 phase/motion，所以立刻释放
    del phase_nocorr
    del motion_nocorr
    del mapped_nocorr
    del opt_nocorr
    cleanup_memory()

    # ------------------------------------------------------------
    # 2. With error correction
    # ------------------------------------------------------------
    opt_corr = make_case_option(
        base_option=option,
        F1062_ref_sample=F1062_ref_sample,
        F1062_single_plane=F1062_single_plane,
        slice_in_crop=slice_in_crop,
        layer=layer,
        wrong_region_enable=True,
    )

    phase_corr, motion_corr, mapped_corr = calFlowCrossResolution.getMotion_v2(
        F1062_single_plane,
        F1062_ref_sample,
        opt_corr,
        verbose=verbose_run,
    )

    mapped_corr_save = to_numpy_cpu(mapped_corr, dtype=np.float32).squeeze()
    loss_corr_roi = roi_mae(F1062_single_plane, mapped_corr_save)

    del phase_corr
    del motion_corr
    del mapped_corr
    del opt_corr
    cleanup_memory()

    delta_loss = loss_corr_roi - loss_nocorr_roi

    print(
        f"  ROI loss: "
        f"nocorr={loss_nocorr_roi:.6f}, "
        f"corr={loss_corr_roi:.6f}, "
        f"delta={delta_loss:.6f}"
    )

    # ------------------------------------------------------------
    # 3. Save images
    # ------------------------------------------------------------
    if save_mapped_tiff:
        moving_save = to_numpy_cpu(F1062_single_plane, dtype=np.float32).squeeze()

        IO.write_multichannel_volume_as_ome_tiff(
            volume=[
                np.ascontiguousarray(mapped_nocorr_save),
                np.ascontiguousarray(mapped_corr_save),
                np.ascontiguousarray(moving_save),
            ],
            out_dir=case_dir,
            frame_idx=0,
            label=f"frame{frame}_slice{raw_slice}_layer{layer}",
        )

        del moving_save
        cleanup_memory()

    # ------------------------------------------------------------
    # 4. Save metrics
    # ------------------------------------------------------------
    np.savez_compressed(
        os.path.join(case_dir, "metrics.npz"),
        frame=frame,
        slice_in_crop=slice_in_crop,
        raw_slice=raw_slice,
        layer=layer,
        loss_nocorr_roi=loss_nocorr_roi,
        loss_corr_roi=loss_corr_roi,
        delta_loss=delta_loss,
    )

    result.update(
        {
            "loss_nocorr_roi": loss_nocorr_roi,
            "loss_corr_roi": loss_corr_roi,
            "delta_loss": delta_loss,
            "status": "ok",
            "error": "",
        }
    )

    del mapped_nocorr_save
    del mapped_corr_save
    cleanup_memory()

    return result


# ------------------------------------------------------------
# Read data
# ------------------------------------------------------------
safe_makedirs(out_root)

cleanup_memory()

print("Reading ND2 data...")

F1062_mov = IO.readND2Frame(
    F1062_path,
    frames=[0, 1, 2, 3, 4],
    channel=1,
).squeeze()

# 原始数据可能是 (T, Z, H, W)，这里转为 (T, H, W, Z)
# 关键修改：transpose 后立刻 ascontiguousarray，避免后续 CUDA 读取非连续内存
F1062_mov = np.ascontiguousarray(
    F1062_mov.transpose(0, 2, 3, 1),
    dtype=np.float32,
)

# Fixed reference
F1062_ref_sample = np.ascontiguousarray(
    F1062_mov[ref_frame, 0:800, 1504:2304, ref_z0:ref_z1],
    dtype=np.float32,
)

print("F1062_mov shape:", F1062_mov.shape, "contiguous:", F1062_mov.flags["C_CONTIGUOUS"])
print("F1062_ref_sample shape:", F1062_ref_sample.shape, "contiguous:", F1062_ref_sample.flags["C_CONTIGUOUS"])

cleanup_memory()


# ------------------------------------------------------------
# Main sweep
# ------------------------------------------------------------
records = []

slices_in_crop = list(slices_in_crop)
total_cases = len(frames_to_run) * len(slices_in_crop) * len(layers_to_run)
case_id = 0

try:
    for frame in frames_to_run:
        # 关键修改：每个 frame 的 crop 也强制连续
        F1062_mov_sample = np.ascontiguousarray(
            F1062_mov[frame, 0:800, 1504:2304, ref_z0:ref_z1],
            dtype=np.float32,
        )

        print(
            f"\nFrame {frame}: "
            f"mov_sample shape={F1062_mov_sample.shape}, "
            f"contiguous={F1062_mov_sample.flags['C_CONTIGUOUS']}"
        )

        for slice_in_crop in slices_in_crop:
            raw_slice = slice_in_crop + ref_z0

            # 关键修改：single plane 强制连续
            F1062_single_plane = np.ascontiguousarray(
                F1062_mov_sample[:, :, slice_in_crop:slice_in_crop + 1],
                dtype=np.float32,
            )

            for layer in layers_to_run:
                case_id += 1

                case_dir = os.path.join(
                    out_root,
                    f"frame_{frame}",
                    f"slice_{raw_slice}",
                    f"layer_{layer}",
                )
                safe_makedirs(case_dir)

                print(
                    f"\n[{case_id}/{total_cases}] "
                    f"frame={frame}, slice_in_crop={slice_in_crop}, "
                    f"raw_slice={raw_slice}, layer={layer}"
                )

                print_mem("  before case")

                try:
                    rec = run_one_case(
                        frame=frame,
                        slice_in_crop=slice_in_crop,
                        raw_slice=raw_slice,
                        layer=layer,
                        case_dir=case_dir,
                        F1062_ref_sample=F1062_ref_sample,
                        F1062_single_plane=F1062_single_plane,
                        option=option,
                    )

                    records.append(rec)

                except Exception as e:
                    err_text = traceback.format_exc()
                    err_repr = repr(e)

                    print(
                        f"  ERROR: frame={frame}, raw_slice={raw_slice}, layer={layer}\n"
                        f"  {err_repr}"
                    )

                    with open(os.path.join(case_dir, "error.txt"), "w", encoding="utf-8") as f:
                        f.write(err_text)

                    records.append(
                        {
                            "frame": frame,
                            "slice_in_crop": slice_in_crop,
                            "raw_slice": raw_slice,
                            "layer": layer,
                            "loss_nocorr_roi": np.nan,
                            "loss_corr_roi": np.nan,
                            "delta_loss": np.nan,
                            "case_dir": case_dir,
                            "status": "error",
                            "error": err_repr,
                        }
                    )

                    save_records(records, out_root)

                    # cudaErrorIllegalAddress 后 CUDA context 通常已经不可恢复
                    if (
                        stop_on_cuda_illegal_address
                        and (
                            "cudaErrorIllegalAddress" in err_text
                            or "illegal memory access" in err_text
                            or "CUDA error: an illegal memory access" in err_text
                        )
                    ):
                        cleanup_memory()
                        raise RuntimeError(
                            "Detected cudaErrorIllegalAddress. "
                            "CUDA context is likely corrupted. "
                            "Please restart the Python kernel/process before continuing. "
                            "The current case has been saved to error.txt."
                        ) from e

                finally:
                    cleanup_memory()
                    print_mem("  after case")

            # 当前 single plane 用完释放
            del F1062_single_plane
            cleanup_memory()

        # 当前 frame crop 用完释放
        del F1062_mov_sample
        cleanup_memory()

finally:
    save_records(records, out_root)
    cleanup_memory()

Reading ND2 data...
F1062_mov shape: (5, 1170, 2304, 45) contiguous: True
F1062_ref_sample shape: (800, 800, 40) contiguous: True

Frame 1: mov_sample shape=(800, 800, 40), contiguous=True

[1/360] frame=1, slice_in_crop=5, raw_slice=10, layer=1
  before case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB
  ROI loss: nocorr=21.045664, corr=20.848949, delta=-0.196714
  after case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB

[2/360] frame=1, slice_in_crop=5, raw_slice=10, layer=2
  before case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB
  ROI loss: nocorr=21.021189, corr=20.822552, delta=-0.198637
  after case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB

[3/360] frame=1, slice_in_crop=5, raw_slice=10, layer=3
  before case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB
  ROI loss: nocorr=20.986610, corr=20.

In [9]:
df = pd.DataFrame(records)

summary_csv = os.path.join(out_root, "summary_all_cases.csv")
df.to_csv(summary_csv, index=False)

# Cases where correction is best than no correction
df_ok = df[df["status"] == "ok"].copy()
df_best = df_ok[df_ok["delta_loss"] < 0].copy()
df_top20_best = df_best.sort_values("delta_loss", ascending=True).head(20)

top20_csv = os.path.join(out_root, "top20_correction_best_cases.csv")
df_top20_best.to_csv(top20_csv, index=False)

print("\n============================================================")
print("Sweep finished.")
print(f"All summary saved to: {summary_csv}")
print(f"Top-20 best cases saved to: {top20_csv}")
print("============================================================")

print("\nTop-20 cases where error correction is best:")
print(
    df_top20_best[
        [
            "frame",
            "raw_slice",
            "slice_in_crop",
            "layer",
            "loss_nocorr_roi",
            "loss_corr_roi",
            "delta_loss",
            "case_dir",
        ]
    ].to_string(index=False)
)


Sweep finished.
All summary saved to: ./HR_exp/F1062_exp_sweep_v3/summary_all_cases.csv
Top-20 best cases saved to: ./HR_exp/F1062_exp_sweep_v3/top20_correction_best_cases.csv

Top-20 cases where error correction is best:
 frame  raw_slice  slice_in_crop  layer  loss_nocorr_roi  loss_corr_roi  delta_loss                                             case_dir
     3         34             29      1        23.029196      22.609537   -0.419659 ./HR_exp/F1062_exp_sweep_v3/frame_3/slice_34/layer_1
     4         34             29      1        23.077253      22.751528   -0.325726 ./HR_exp/F1062_exp_sweep_v3/frame_4/slice_34/layer_1
     4         37             32      1        23.039362      22.738073   -0.301289 ./HR_exp/F1062_exp_sweep_v3/frame_4/slice_37/layer_1
     4         25             20      3        22.390022      22.096792   -0.293230 ./HR_exp/F1062_exp_sweep_v3/frame_4/slice_25/layer_3
     4         33             28      1        22.875015      22.591331   -0.283684 ./HR_exp

option: slice 34 layer 1  frame3

F2013
---

In [ ]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
import os
import gc
import csv
import traceback
import numpy as np

# 如果你后面要用 pandas，可以保留；没有 pandas 也不影响主流程
try:
    import pandas as pd
except Exception:
    pd = None


# ------------------------------------------------------------
# Basic config
# ------------------------------------------------------------
out_root = "./HR_exp/f2013_exp_sweep_v2"

f2013_path = "/home/cyf/wbi/Virginia/raw_data/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15850.nd2"

frames_to_run = [1,2]
slices_in_crop = range(3, 16)   # 5..24 inclusive
layers_to_run = [0,1,2]

ref_frame = 0
ref_z0 = 0
ref_z1 = 19

roi_x0, roi_x1 = 0, 1152
roi_y0, roi_y1 = 0,1152

verbose_run = False

# 遇到 cudaErrorIllegalAddress 时建议直接停止，因为 CUDA context 往往已经坏掉
stop_on_cuda_illegal_address = True

# 是否保存 mapped 图像
save_mapped_tiff = True


# ------------------------------------------------------------
# Memory helpers
# ------------------------------------------------------------
def cleanup_memory():
    """
    Release Python / PyTorch / CuPy cached memory as much as possible.
    """
    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

    try:
        import cupy as cp
        cp.cuda.Stream.null.synchronize()
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass


def to_numpy_cpu(x, dtype=np.float32):
    """
    Convert numpy / cupy / torch tensor to CPU numpy array.
    """
    try:
        import cupy as cp
        if isinstance(x, cp.ndarray):
            x = cp.asnumpy(x)
    except Exception:
        pass

    try:
        import torch
        if torch.is_tensor(x):
            x = x.detach().cpu().numpy()
    except Exception:
        pass

    x = np.asarray(x)

    if dtype is not None:
        x = x.astype(dtype, copy=False)

    return np.ascontiguousarray(x)


def safe_delete(*names):
    """
    This function is only a placeholder for readability.
    Real deletion is done by del variable_name in the caller scope.
    """
    cleanup_memory()


def print_mem(prefix=""):
    """
    Optional GPU memory print. Useful for debugging memory leakage.
    """
    msg = prefix

    try:
        import torch
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            msg += f" | torch allocated={alloc:.3f} GB, reserved={reserved:.3f} GB"
    except Exception:
        pass

    try:
        import cupy as cp
        pool = cp.get_default_memory_pool()
        used = pool.used_bytes() / 1024**3
        total = pool.total_bytes() / 1024**3
        msg += f" | cupy used={used:.3f} GB, pool={total:.3f} GB"
    except Exception:
        pass

    if msg:
        print(msg)


# ------------------------------------------------------------
# IO helpers
# ------------------------------------------------------------
def safe_makedirs(path):
    os.makedirs(path, exist_ok=True)


def save_records(records, out_root):
    safe_makedirs(out_root)

    csv_path = os.path.join(out_root, "summary.csv")

    if len(records) == 0:
        return

    fieldnames = list(records[0].keys())

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)

    if pd is not None:
        df = pd.DataFrame(records)
        df.to_csv(csv_path, index=False)

        ok_df = df[df["status"] == "ok"].copy()
        if len(ok_df) > 0:
            ok_df = ok_df.sort_values("delta_loss", ascending=False)
            top20_path = os.path.join(out_root, "top20_corr_worse_than_nocorr.csv")
            ok_df.head(20).to_csv(top20_path, index=False)

    print(f"\nSaved summary to: {csv_path}")


# ------------------------------------------------------------
# Metric
# ------------------------------------------------------------
def roi_mae(src, pred):
    """
    Compute MAE inside ROI.
    Both src and pred can be shape (H, W, 1) or (H, W).
    """
    src2 = to_numpy_cpu(src, dtype=np.float32).squeeze()
    pred2 = to_numpy_cpu(pred, dtype=np.float32).squeeze()

    src_roi = src2[roi_x0:roi_x1, roi_y0:roi_y1]
    pred_roi = pred2[roi_x0:roi_x1, roi_y0:roi_y1]

    return float(np.mean(np.abs(src_roi - pred_roi)))


# ------------------------------------------------------------
# Option builder
# ------------------------------------------------------------
def make_case_option(
    base_option,
    f2013_ref_sample,
    f2013_single_plane,
    slice_in_crop,
    layer,
    wrong_region_enable,
):
    """
    Build an independent option dict for one case.

    Important changes:
    1. Use shallow copy instead of deepcopy.
    2. Remove stale large fields from base_option.
    3. Force phase / masks to be C-contiguous CPU numpy arrays.
    4. Avoid np.indices to reduce temporary memory.
    """

    # 不建议 deepcopy，因为 base_option 里可能带有旧的 phase/motion/cache/GPU 对象
    opt = dict(base_option)

    # 删除可能残留的大对象，避免跨 case 污染
    for k in [
        "phase",
        "motion",
        "mask_ref",
        "mask_mov",
        "mapped",
        "mapped_ref",
        "mapped_mov",
        "cache",
        "cached_ref",
        "cached_mov",
        "gpu_cache",
    ]:
        opt.pop(k, None)

    opt["motion"] = None
    opt["r"] = 5
    opt["layer"] = int(layer)
    opt["iter"] = 10
    opt["movRange"] = 15.0
    opt["tol"] = 1e-6

    opt["zRatio"] = 1
    opt["zRatio_HR"] = 3

    # mask 强制连续
    opt["mask_ref"] = np.ascontiguousarray(
        np.zeros(f2013_ref_sample.shape, dtype=bool)
    )
    opt["mask_mov"] = np.ascontiguousarray(
        np.zeros(f2013_single_plane.shape, dtype=bool)
    )

    smoothPenalty_raw = 0.005
    Pnltfactor = prep.getSmPnltNormFctr(f2013_ref_sample, opt)
    opt["smoothPenalty"] = Pnltfactor * smoothPenalty_raw

    # phase init
    # 不使用 np.indices，减少 int64 临时数组
    H, W, D = f2013_single_plane.shape

    phase_init = np.empty((H, W, D, 3), dtype=np.float32)

    x_coord = np.arange(H, dtype=np.float32)[:, None, None]
    y_coord = np.arange(W, dtype=np.float32)[None, :, None]
    z_coord = np.arange(D, dtype=np.float32)[None, None, :]

    phase_init[..., 0] = x_coord
    phase_init[..., 1] = y_coord
    phase_init[..., 2] = (
        float(slice_in_crop)
        + z_coord * float(opt["zRatio"]) / float(opt["zRatio_HR"])
    )

    opt["phase"] = np.ascontiguousarray(phase_init, dtype=np.float32)
    opt["motion"] = None

    opt["wrong_region_enable"] = bool(wrong_region_enable)

    if wrong_region_enable:
        opt["wrong_region_metric"] = "mae"
        opt["wrong_region_mad_threshold"] = 3.5
        opt["wrong_region_min_component_size"] = 2
        opt["wrong_region_exclude_mode"] = "highresidual"

        opt["motion_inpaint_enable"] = True
        opt["motion_inpaint_iters"] = 20
        opt["motion_inpaint_neighbor_mode"] = "4"
    else:
        # 避免 base_option 里原本残留这些开关
        opt["motion_inpaint_enable"] = False

    return opt


# ------------------------------------------------------------
# Run one case
# ------------------------------------------------------------
def run_one_case(
    frame,
    slice_in_crop,
    raw_slice,
    layer,
    case_dir,
    f2013_ref_sample,
    f2013_single_plane,
    option,
):
    """
    Run no-correction and correction sequentially.
    The no-correction large variables are released before correction starts.
    """

    result = {
        "frame": frame,
        "slice_in_crop": slice_in_crop,
        "raw_slice": raw_slice,
        "layer": layer,
        "loss_nocorr_roi": np.nan,
        "loss_corr_roi": np.nan,
        "delta_loss": np.nan,
        "case_dir": case_dir,
        "status": "error",
        "error": "",
    }

    mapped_nocorr_save = None
    mapped_corr_save = None

    # ------------------------------------------------------------
    # 1. No correction
    # ------------------------------------------------------------
    opt_nocorr = make_case_option(
        base_option=option,
        f2013_ref_sample=f2013_ref_sample,
        f2013_single_plane=f2013_single_plane,
        slice_in_crop=slice_in_crop,
        layer=layer,
        wrong_region_enable=False,
    )

    phase_nocorr, motion_nocorr, mapped_nocorr = calFlowCrossResolution.getMotion_v2(
        f2013_single_plane,
        f2013_ref_sample,
        opt_nocorr,
        verbose=verbose_run,
    )

    mapped_nocorr_save = to_numpy_cpu(mapped_nocorr, dtype=np.float32).squeeze()
    loss_nocorr_roi = roi_mae(f2013_single_plane, mapped_nocorr_save)

    # 不保存 phase/motion，所以立刻释放
    del phase_nocorr
    del motion_nocorr
    del mapped_nocorr
    del opt_nocorr
    cleanup_memory()

    # ------------------------------------------------------------
    # 2. With error correction
    # ------------------------------------------------------------
    opt_corr = make_case_option(
        base_option=option,
        f2013_ref_sample=f2013_ref_sample,
        f2013_single_plane=f2013_single_plane,
        slice_in_crop=slice_in_crop,
        layer=layer,
        wrong_region_enable=True,
    )

    phase_corr, motion_corr, mapped_corr = calFlowCrossResolution.getMotion_v2(
        f2013_single_plane,
        f2013_ref_sample,
        opt_corr,
        verbose=verbose_run,
    )

    mapped_corr_save = to_numpy_cpu(mapped_corr, dtype=np.float32).squeeze()
    loss_corr_roi = roi_mae(f2013_single_plane, mapped_corr_save)

    del phase_corr
    del motion_corr
    del mapped_corr
    del opt_corr
    cleanup_memory()

    delta_loss = loss_corr_roi - loss_nocorr_roi

    print(
        f"  ROI loss: "
        f"nocorr={loss_nocorr_roi:.6f}, "
        f"corr={loss_corr_roi:.6f}, "
        f"delta={delta_loss:.6f}"
    )

    # ------------------------------------------------------------
    # 3. Save images
    # ------------------------------------------------------------
    if save_mapped_tiff:
        moving_save = to_numpy_cpu(f2013_single_plane, dtype=np.float32).squeeze()

        IO.write_multichannel_volume_as_ome_tiff(
            volume=[
                np.ascontiguousarray(mapped_nocorr_save),
                np.ascontiguousarray(mapped_corr_save),
                np.ascontiguousarray(moving_save),
            ],
            out_dir=case_dir,
            frame_idx=0,
            label=f"frame{frame}_slice{raw_slice}_layer{layer}",
        )

        del moving_save
        cleanup_memory()

    # ------------------------------------------------------------
    # 4. Save metrics
    # ------------------------------------------------------------
    np.savez_compressed(
        os.path.join(case_dir, "metrics.npz"),
        frame=frame,
        slice_in_crop=slice_in_crop,
        raw_slice=raw_slice,
        layer=layer,
        loss_nocorr_roi=loss_nocorr_roi,
        loss_corr_roi=loss_corr_roi,
        delta_loss=delta_loss,
    )

    result.update(
        {
            "loss_nocorr_roi": loss_nocorr_roi,
            "loss_corr_roi": loss_corr_roi,
            "delta_loss": delta_loss,
            "status": "ok",
            "error": "",
        }
    )

    del mapped_nocorr_save
    del mapped_corr_save
    cleanup_memory()

    return result


# ------------------------------------------------------------
# Read data
# ------------------------------------------------------------
safe_makedirs(out_root)

cleanup_memory()

print("Reading ND2 data...")

f2013_mov = IO.readND2Frame(
    f2013_path,
    frames=[0, 1, 2],
    channel=1,
).squeeze()

# 原始数据可能是 (T, Z, H, W)，这里转为 (T, H, W, Z)
# 关键修改：transpose 后立刻 ascontiguousarray，避免后续 CUDA 读取非连续内存
f2013_mov = np.ascontiguousarray(
    f2013_mov.transpose(0, 2, 3, 1),
    dtype=np.float32,
)

# Fixed reference
f2013_ref_sample = np.ascontiguousarray(
    f2013_mov[ref_frame, :, :, ref_z0:ref_z1],
    dtype=np.float32,
)

print("f2013_mov shape:", f2013_mov.shape, "contiguous:", f2013_mov.flags["C_CONTIGUOUS"])
print("f2013_ref_sample shape:", f2013_ref_sample.shape, "contiguous:", f2013_ref_sample.flags["C_CONTIGUOUS"])

cleanup_memory()


# ------------------------------------------------------------
# Main sweep
# ------------------------------------------------------------
records = []

slices_in_crop = list(slices_in_crop)
total_cases = len(frames_to_run) * len(slices_in_crop) * len(layers_to_run)
case_id = 0

try:
    for frame in frames_to_run:
        # 关键修改：每个 frame 的 crop 也强制连续
        f2013_mov_sample = np.ascontiguousarray(
            f2013_mov[frame, :, :, ref_z0:ref_z1],
            dtype=np.float32,
        )

        print(
            f"\nFrame {frame}: "
            f"mov_sample shape={f2013_mov_sample.shape}, "
            f"contiguous={f2013_mov_sample.flags['C_CONTIGUOUS']}"
        )

        for slice_in_crop in slices_in_crop:
            raw_slice = slice_in_crop + ref_z0

            # 关键修改：single plane 强制连续
            f2013_single_plane = np.ascontiguousarray(
                f2013_mov_sample[:, :, slice_in_crop:slice_in_crop + 1],
                dtype=np.float32,
            )

            for layer in layers_to_run:
                case_id += 1

                case_dir = os.path.join(
                    out_root,
                    f"frame_{frame}",
                    f"slice_{raw_slice}",
                    f"layer_{layer}",
                )
                safe_makedirs(case_dir)

                print(
                    f"\n[{case_id}/{total_cases}] "
                    f"frame={frame}, slice_in_crop={slice_in_crop}, "
                    f"raw_slice={raw_slice}, layer={layer}"
                )

                print_mem("  before case")

                try:
                    rec = run_one_case(
                        frame=frame,
                        slice_in_crop=slice_in_crop,
                        raw_slice=raw_slice,
                        layer=layer,
                        case_dir=case_dir,
                        f2013_ref_sample=f2013_ref_sample,
                        f2013_single_plane=f2013_single_plane,
                        option=option,
                    )

                    records.append(rec)

                except Exception as e:
                    err_text = traceback.format_exc()
                    err_repr = repr(e)

                    print(
                        f"  ERROR: frame={frame}, raw_slice={raw_slice}, layer={layer}\n"
                        f"  {err_repr}"
                    )

                    with open(os.path.join(case_dir, "error.txt"), "w", encoding="utf-8") as f:
                        f.write(err_text)

                    records.append(
                        {
                            "frame": frame,
                            "slice_in_crop": slice_in_crop,
                            "raw_slice": raw_slice,
                            "layer": layer,
                            "loss_nocorr_roi": np.nan,
                            "loss_corr_roi": np.nan,
                            "delta_loss": np.nan,
                            "case_dir": case_dir,
                            "status": "error",
                            "error": err_repr,
                        }
                    )

                    save_records(records, out_root)

                    # cudaErrorIllegalAddress 后 CUDA context 通常已经不可恢复
                    if (
                        stop_on_cuda_illegal_address
                        and (
                            "cudaErrorIllegalAddress" in err_text
                            or "illegal memory access" in err_text
                            or "CUDA error: an illegal memory access" in err_text
                        )
                    ):
                        cleanup_memory()
                        raise RuntimeError(
                            "Detected cudaErrorIllegalAddress. "
                            "CUDA context is likely corrupted. "
                            "Please restart the Python kernel/process before continuing. "
                            "The current case has been saved to error.txt."
                        ) from e

                finally:
                    cleanup_memory()
                    print_mem("  after case")

            # 当前 single plane 用完释放
            del f2013_single_plane
            cleanup_memory()

        # 当前 frame crop 用完释放
        del f2013_mov_sample
        cleanup_memory()

finally:
    save_records(records, out_root)
    cleanup_memory()

Reading ND2 data...
f2013_mov shape: (3, 1152, 1152, 19) contiguous: True
f2013_ref_sample shape: (1152, 1152, 19) contiguous: True

Frame 1: mov_sample shape=(1152, 1152, 19), contiguous=True

[1/52] frame=1, slice_in_crop=3, raw_slice=3, layer=1
  before case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB
  ROI loss: nocorr=25.891077, corr=25.888281, delta=-0.002796
  after case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB

[2/52] frame=1, slice_in_crop=3, raw_slice=3, layer=2
  before case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB
  ROI loss: nocorr=25.911243, corr=25.915199, delta=0.003956
  after case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB

[3/52] frame=1, slice_in_crop=4, raw_slice=4, layer=1
  before case | torch allocated=0.000 GB, reserved=0.000 GB | cupy used=0.000 GB, pool=0.000 GB
  ROI loss: nocorr=25.933832, corr=25.926

In [6]:
df = pd.DataFrame(records)

summary_csv = os.path.join(out_root, "summary_all_cases.csv")
df.to_csv(summary_csv, index=False)

# Cases where correction is best than no correction
df_ok = df[df["status"] == "ok"].copy()
df_best = df_ok[df_ok["delta_loss"] < 0].copy()
df_top20_best = df_best.sort_values("delta_loss", ascending=True).head(20)

top20_csv = os.path.join(out_root, "top20_correction_best_cases.csv")
df_top20_best.to_csv(top20_csv, index=False)

print("\n============================================================")
print("Sweep finished.")
print(f"All summary saved to: {summary_csv}")
print(f"Top-20 best cases saved to: {top20_csv}")
print("============================================================")

print("\nTop-20 cases where error correction is best:")
print(
    df_top20_best[
        [
            "frame",
            "raw_slice",
            "slice_in_crop",
            "layer",
            "loss_nocorr_roi",
            "loss_corr_roi",
            "delta_loss",
            "case_dir",
        ]
    ].to_string(index=False)
)


Sweep finished.
All summary saved to: ./HR_exp/f2013_exp_sweep/summary_all_cases.csv
Top-20 best cases saved to: ./HR_exp/f2013_exp_sweep/top20_correction_best_cases.csv

Top-20 cases where error correction is best:
 frame  raw_slice  slice_in_crop  layer  loss_nocorr_roi  loss_corr_roi  delta_loss                                          case_dir
     1         11             11      2        26.103790      25.952906   -0.150885 ./HR_exp/f2013_exp_sweep/frame_1/slice_11/layer_2
     1          7              7      2        26.031370      25.887955   -0.143415  ./HR_exp/f2013_exp_sweep/frame_1/slice_7/layer_2
     2         15             15      1        26.268730      26.126829   -0.141901 ./HR_exp/f2013_exp_sweep/frame_2/slice_15/layer_1
     2         15             15      2        26.294590      26.158640   -0.135950 ./HR_exp/f2013_exp_sweep/frame_2/slice_15/layer_2
     1          5              5      1        25.960138      25.948481   -0.011658  ./HR_exp/f2013_exp_sweep/fra

F338
---